In [ ]:
%%capture
!pip install keras-cv

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow" # other options: tensorflow or torch
import json
import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 47  # Random seed
    neg_sample = 0.01 # Downsample negative calss
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [128, 128]  # Input image size
    epochs = 10 # Training epochs
    batch_size = 256  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
ROOT = str(os.getcwd())
BASE_PATH = f"{ROOT}/drive/MyDrive/WAIT"
DATA_PATH = f"{BASE_PATH}/data/new"
MODEL_PATH = f"{BASE_PATH}/models/enhanced/"
PROC_PATH = f"{MODEL_PATH}/enhanced/preprocessed_dataset/"

In [ ]:
# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet').ffill()

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet').ffill()

In [ ]:
print("Class Distribution Before Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

# Sampling
positive_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
negative_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([positive_df, negative_df], axis=0).sample(frac=1.0)

print("\nCalss Distribution After Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

Class Distribution Before Sampling (%):


,proportion
target,
0,99.899858
1,0.100142



Calss Distribution After Sampling (%):


,proportion
target,
0,66.611631
1,33.388369


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(0.7506196640044065), 1: np.float64(1.4975274725274725)}


In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

# Num Train: 4383 | Num Valid: 1068


In [ ]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

training:
target
0    2929
1    1454
Name: count, dtype: int64

validation:
target
0    702
1    366
Name: count, dtype: int64


# 📊 | Tabular Features

### Enhanced


In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS =["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    # Define augmentations
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]

    # Apply augmentations to random samples
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]

    # Build augmentation layer
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    # Apply augmentations
    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        # Read jpeg image
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)

        # Resize
        image = tf.image.resize(image, size=target_size, method="area")

        # Rescale image
        image = tf.cast(image, tf.float32)
        image /= 255.0

        # Reshape
        image = tf.reshape(image, [*target_size, 3])

        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.experimental.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds
    ds = ds.prefetch(AUTO)
    return ds

In [ ]:
# Build Training & Validation Dataset

print("# Training:")
training_features = dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(training_ids, training_validation_hdf5, training_features,
                         training_labels, batch_size=CFG.batch_size,
                         shuffle=True, augment=True)

print("# Validation:")
validation_features = dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(validation_ids, training_validation_hdf5, validation_features,
                         validation_labels, batch_size=CFG.batch_size,
                         shuffle=False, augment=False)


# Training:


Loading Images :   0%|          | 0/4383 [00:00<?, ?it/s]

# Validation:


Loading Images :   0%|          | 0/1068 [00:00<?, ?it/s]

In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

In [ ]:
training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

(None, 128)


In [ ]:
# fs_dir = os.path.join(MODEL_PATH, "model-metadata")
# os.makedirs(fs_dir, exist_ok=True)
# filepath = os.path.join(fs_dir, "feature_space_test.keras")

# feature_space.save(filepath)
# print(f"FeatureSpace saved at: '{filepath}'")

In [ ]:
# Apply Feature Processing

training_ds = training_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

validation_ds = validation_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

# Testing
print("# Testing:")
testing_features = dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(testing_ids, testing_hdf5,
                        testing_features, batch_size=CFG.batch_size,
                         shuffle=False, augment=False, cache=False)

# Apply feature space processing
testing_ds = testing_ds.map(
    lambda x: {"images": x["images"],
               "features": feature_space(x["features"])}, num_parallel_calls=tf.data.AUTOTUNE)

# Testing:


Loading Images :   0%|          | 0/37576 [00:00<?, ?it/s]

In [ ]:
# loss & metric
auc = keras.metrics.AUC()
loss = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss,
    metrics=[auc],
)

# Model Summary
model.summary()

100%|██████████| 1.79k/1.79k [00:00<00:00, 2.73MB/s]


100%|██████████| 34.2M/34.2M [00:01<00:00, 29.3MB/s]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer) │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features            │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficient_net_v2b2… │ (None, 4, 4,      │  8,769,374 │ images[0][0]      │
│ (EfficientNetV2Bac… │ 1408)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 96)        │     12,384 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1408)      │          0 │ efficient_net_v2… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     12,416 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1408)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1536)      │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │      1,537 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,795,711 (33.55 MB)

 Trainable params: 8,713,423 (33.24 MB)

 Non-trainable params: 82,288 (321.44 KB)

In [ ]:
# keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, dpi=60)

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

In [ ]:
asf

In [ ]:
%%capture
!pip install --no-cache-dir tf-explain
!pip install shap

In [ ]:
# tf.keras.backend.clear_session()

In [ ]:
_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

In [ ]:
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1] == 128

per-feature dim sizes: {'sex': 3, 'anatom_site_general': 6, 'tbp_tile_type': 3, 'tbp_lv_location': 20, 'combined_anatomical_site': 24, 'attribution': 8, 'age_approx': 32, 'tbp_lv_nevi_confidence': 1, 'clin_size_long_diam_mm': 1, 'tbp_lv_areaMM2': 1, 'tbp_lv_area_perim_ratio': 1, 'tbp_lv_color_std_mean': 1, 'tbp_lv_deltaLBnorm': 1, 'tbp_lv_minorAxisMM': 1, 'lesion_size_ratio': 1, 'lesion_shape_index': 1, 'hue_contrast': 1, 'luminance_contrast': 1, 'lesion_color_difference': 1, 'border_complexity': 1, '3d_position_distance': 1, 'perimeter_to_area_ratio': 1, 'lesion_visibility_score': 1, 'symmetry_border_consistency': 1, 'color_consistency': 1, 'size_age_interaction': 1, 'hue_color_std_interaction': 1, 'lesion_severity_index': 1, 'shape_complexity_index': 1, 'color_contrast_index': 1, 'log_lesion_area': 1, 'mean_hue_difference': 1, 'std_dev_contrast': 1, 'color_shape_composite_index': 1, '3d_lesion_orientation': 1, 'overall_color_difference': 1, 'symmetry_perimeter_interaction': 1, 'compr

True

In [ ]:
def aggregate_raw_importance(encoded_feature_names, mean_abs_vec):
    agg = {}
    for j, val in enumerate(mean_abs_vec):
        name = encoded_feature_names[j]
        if "__" in name:
            raw = name.split("__", 1)[0]
        elif name.startswith("age_approx_bin_"):
            raw = "age_approx"
        else:
            raw = name
        agg.setdefault(raw, 0.0)
        agg[raw] += float(abs(val))
    return agg

In [58]:
import seaborn as sns

def plot_topk_grouped_raw_global_pos_neg(
    global_imp, pos_imp, neg_imp, k=10, epoch_num=None, figsize=(10, 5), out_path=None
):
    # Ranking
    items = sorted(global_imp.items(), key=lambda kv: kv[1], reverse=True)[:k]
    names = [k for k, _ in items]

    df = pd.DataFrame({
        "feature": names,
        "Global":   [global_imp.get(n, 0.0) for n in names],
        "Pos preds":[pos_imp.get(n,   0.0) for n in names],
        "Neg preds":[neg_imp.get(n,   0.0) for n in names],
    })

    df_melt = df.melt(id_vars="feature", var_name="Class", value_name="Importance")

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt,
        y="feature", x="Importance", hue="Class",
        palette=(
            sns.color_palette("bright")[4],  # Global
            sns.color_palette("bright")[0],  # Pos
            sns.color_palette("bright")[3],  # Neg
        ),
        edgecolor="black",
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="{:.2f}", padding=4, fontsize=7)

    plt.ylabel("Mean |SHAP|")
    plt.xlabel("")
    title = f"Feature Importance (top {k}) – Global vs Pos vs Neg"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    plt.title(title)
    plt.tight_layout()

    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
def encoded_indices_for_raw(encoded_feature_names, feat_name):
    idxs = []
    for j, name in enumerate(encoded_feature_names):
        if "__" in name:
            if name.split("__", 1)[0] == feat_name:
                idxs.append(j)
        elif name.startswith(f"{feat_name}_bin_"):
            idxs.append(j)
        elif name == feat_name:
            idxs.append(j)
    return np.array(idxs, dtype=int)


def select_top_numeric_features(global_importance_raw, raw_batch_df, k=4):
    """Return top-k raw features among those that are numeric-only."""
    numeric_feats = [
        f for f in global_importance_raw.keys()
        if f in raw_batch_df.columns and raw_batch_df[f].dtype.kind in "if"  # int or float
    ]

    ordered = sorted(
        numeric_feats,
        key=lambda name: global_importance_raw[name],
        reverse=True,
    )

    return ordered[:k]

In [59]:
def plot_shap_scatter_top4_numeric(
    shap_vals,
    preds,
    encoded_feature_names,
    global_importance_raw,
    raw_batch_df,
    explained_indices,
    epoch_num=None,
    positive_threshold=0.5,
    out_path=None,
):
    top_feats = select_top_numeric_features(global_importance_raw, raw_batch_df, k=4)

    pred_scores = preds[:, 0]
    pos_mask = pred_scores >= positive_threshold
    neg_mask = ~pos_mask

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    title = "SHAP values vs. top numeric features (pos vs. neg predictions)"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    fig.suptitle(title, fontsize=14)

    colors = sns.color_palette("bright", 5)
    pos_color, neg_color = colors[0], colors[3]

    for i, (ax, feat_name) in enumerate(zip(axes, top_feats)):
        enc_idx = encoded_indices_for_raw(encoded_feature_names, feat_name)
        if enc_idx.size == 0:
            ax.set_title(f"{feat_name} (no encoded dims)")
            ax.axis("off")
            continue

        shap_feat = shap_vals[:, enc_idx].sum(axis=1)

        raw_vals_batch = raw_batch_df[feat_name].values
        raw_vals = raw_vals_batch[explained_indices]

        x_pos = raw_vals[pos_mask]
        y_pos = shap_feat[pos_mask]
        x_neg = raw_vals[neg_mask]
        y_neg = shap_feat[neg_mask]

        ax.scatter(
            x_pos, y_pos,
            s=18, marker="o", alpha=0.7,
            label="Positive", color=pos_color, edgecolors="none",
        )
        ax.scatter(
            x_neg, y_neg,
            s=18, marker="o", alpha=0.8,
            label="Negative", color=neg_color, edgecolors="none",
        )

        ax.axhline(0.0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
        ax.set_title(feat_name)

        if i in (0, 2):
            ax.set_ylabel("SHAP value")

    for i in range(len(top_feats), 4):
        axes[i].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

In [53]:
import shap
from typing import Tuple, Dict, Any

class ShapEngine:
    """
    SHAP engine for the tabular 'features' input of the multimodal model.

    - Original model signature: model({"images": images, "features": features}) -> (batch, 1)
    - Internal wrapper: wrapped_model([features, images]) -> (batch, 1)
    - Explainer: shap.GradientExplainer on wrapped_model
    - We keep SHAP values only for the 'features' input (index 0 in [features, images]).
    """

    def __init__(
        self,
        background_size: int = 128,
        max_explain_samples: int = 64,
        top_k: int = 10,
        positive_threshold: float = 0.5,
        use_gradient_explainer: bool = True,
    ):
        self.background_size = background_size
        self.max_explain_samples = max_explain_samples
        self.top_k = top_k
        self.positive_threshold = positive_threshold
        self.use_gradient_explainer = use_gradient_explainer

        self._explainer = None
        self._wrapped_model = None   # Keras model: [features, images] -> original_model(...)
        self._feature_input_index = 0

    # ----------------- internal utils ----------------- #
    def _build_wrapped_model(
        self,
        base_model: tf.keras.Model,
        feature_shape,
        image_shape,
    ):
        """
        Build a Keras model that takes [features, images] and calls the original
        dict-based model internally.
        """
        feat_in = tf.keras.Input(shape=feature_shape[1:], name="shap_features")
        img_in = tf.keras.Input(shape=image_shape[1:], name="shap_images")

        out = base_model({"features": feat_in, "images": img_in})
        self._wrapped_model = tf.keras.Model(
            inputs=[feat_in, img_in],
            outputs=out,
            name="wrapped_for_shap",
        )

    def _build_explainer(
        self,
        base_model: tf.keras.Model,
        bg_features: np.ndarray,
        bg_images: np.ndarray,
    ):
        # Build wrapped model lazily
        if self._wrapped_model is None:
            self._build_wrapped_model(
                base_model,
                feature_shape=bg_features.shape,
                image_shape=bg_images.shape,
            )

        bg_inputs = [bg_features, bg_images]  # [features, images]

        if self.use_gradient_explainer:
            self._explainer = shap.GradientExplainer(self._wrapped_model, bg_inputs)
        else:
            self._explainer = shap.explainers.Deep(self._wrapped_model, bg_inputs)

    def _select_background_and_expl_set(
        self,
        features: np.ndarray,
        images: np.ndarray,
    ):
        n = features.shape[0]
        bg_idx = np.random.choice(n, size=min(self.background_size, n), replace=False)

        ex_idx = np.arange(n)
        if n > self.max_explain_samples:
            ex_idx = np.random.choice(n, size=self.max_explain_samples, replace=False)

        bg_features = features[bg_idx]
        bg_images = images[bg_idx]
        ex_features = features[ex_idx]
        ex_images = images[ex_idx]

        return (bg_features, bg_images), (ex_features, ex_images), ex_idx

    # ----------------- public API ----------------- #
    def explain(
        self,
        validation_data: Tuple[np.ndarray, np.ndarray, np.ndarray],
        model: tf.keras.Model,
        class_index: int = 0,  # irrelevant for scalar output
    ) -> Tuple[Dict[str, Any], np.ndarray, np.ndarray]:
        """
        validation_data: (features, images, labels) (all numpy arrays)

        Returns:
            stats: dict with
                - mean_abs_all: (n_features,)
                - mean_abs_pos: (n_features,)
                - mean_abs_neg: (n_features,)
                - topk_all_indices, topk_pos_indices, topk_neg_indices
                - explained_indices: indices into the original batch
            shap_values_features: (n_explain, n_features)
            preds: (n_explain, 1)
        """

        features, images, labels = validation_data

        (bg_features, bg_images), (ex_features, ex_images), ex_idx = \
            self._select_background_and_expl_set(features, images)

        # Lazy explainer init on wrapped model
        if self._explainer is None:
            self._build_explainer(model, bg_features, bg_images)

        ex_inputs = [ex_features, ex_images]  # [features, images]
        preds = self._wrapped_model(ex_inputs).numpy()  # (n_ex, 1)

        # SHAP values: list [phi_features, phi_images]
        shap_values_all_inputs = self._explainer.shap_values(ex_inputs)
        shap_values_features = shap_values_all_inputs[self._feature_input_index]
        # (n_explain, n_features)

        shap_values_features = np.array(shap_values_features)

        # Handle shapes:
        # - expected: (n_ex, n_features)
        # - you currently get: (n_ex, n_features, 1)
        if shap_values_features.ndim == 3:
          # If last dim is 1 → (n_ex, n_features, 1) → squeeze last
          if shap_values_features.shape[-1] == 1:
              shap_values_features = shap_values_features[..., 0]
          # If second dim is 1 → (n_ex, 1, n_features) → squeeze middle
          elif shap_values_features.shape[1] == 1:
              shap_values_features = shap_values_features[:, 0, :]
        # Now: (n_ex, n_features)

        # Global mean |SHAP|
        mean_abs_all = np.mean(np.abs(shap_values_features), axis=0)  # (n_features,)

        # Positive / negative groups
        pred_scores = preds[:, 0]
        pos_mask = pred_scores >= self.positive_threshold
        neg_mask = ~pos_mask

        if pos_mask.any():
            mean_abs_pos = np.mean(np.abs(shap_values_features[pos_mask]), axis=0)
        else:
            mean_abs_pos = np.zeros_like(mean_abs_all)

        if neg_mask.any():
            mean_abs_neg = np.mean(np.abs(shap_values_features[neg_mask]), axis=0)
        else:
            mean_abs_neg = np.zeros_like(mean_abs_all)

        def topk_indices(x: np.ndarray, k: int):
            # robust 1D top-k
            x = x.reshape(-1)
            k = min(k, x.shape[0])
            return np.argsort(x)[::-1][:k]

        topk_all_idx = topk_indices(mean_abs_all, self.top_k)
        topk_pos_idx = topk_indices(mean_abs_pos, self.top_k)
        topk_neg_idx = topk_indices(mean_abs_neg, self.top_k)

        stats = {
            "mean_abs_all": mean_abs_all,
            "mean_abs_pos": mean_abs_pos,
            "mean_abs_neg": mean_abs_neg,
            "topk_all_indices": topk_all_idx,
            "topk_pos_indices": topk_pos_idx,
            "topk_neg_indices": topk_neg_idx,
            "explained_indices": ex_idx,
        }

        return stats, shap_values_features, preds

In [60]:
from pathlib import Path
from typing import List

import numpy as np
import tensorflow as tf
from keras.callbacks import Callback


class ShapCallback(Callback):
    """
    Runs ShapEngine every N epochs on a frozen validation subset and produces:
      - grouped barplot (Global vs Pos vs Neg) for top-k raw features
      - 2x2 scatter for top-4 numeric raw features (pos vs neg predictions)

    Explanation set is built at on_train_begin by concatenating multiple
    batches from `data_source`, up to `max_samples` samples (and optionally
    up to `max_batches` batches).
    """

    def __init__(
        self,
        data_source: tf.data.Dataset,
        validation_df,
        feat_cols: List[str],
        encoded_feature_names: List[str],
        base_dir: str,
        background_size: int = 256,
        max_explain_samples: int = 256,
        top_k: int = 15,
        positive_threshold: float = 0.5,
        every_n_epochs: int = 1,
        max_samples: int = 512,   # total samples in explanation set
        max_batches: int | None = None,  # limit num batches to read, or None
        rng_seed: int = 0,
    ):
        super().__init__()
        self.data_source = data_source
        self.validation_df = validation_df
        self.feat_cols = feat_cols
        self.encoded_feature_names = encoded_feature_names

        self.background_size = int(background_size)
        self.max_explain_samples = int(max_explain_samples)
        self.top_k = int(top_k)
        self.positive_threshold = float(positive_threshold)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.max_samples = int(max_samples)
        self.max_batches = max_batches
        self.rng_seed = int(rng_seed)

        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

        # frozen explanation set
        self._images = None
        self._features = None
        self._labels = None
        self._raw_batch_df = None

        # ShapEngine instance
        self._engine = ShapEngine(
            background_size=self.background_size,
            max_explain_samples=self.max_explain_samples,
            top_k=self.top_k,
            positive_threshold=self.positive_threshold,
        )

    # ---------- helpers ---------- #
    def _freeze_batch_from_dataset(self, ds: tf.data.Dataset):
        """
        Build explanation set by concatenating multiple batches from ds.

        Assumes ds yields (inputs, labels) where inputs is {'images','features'}.
        Also builds the corresponding raw_batch_df from validation_df (shuffle=False).
        """
        X_img_list, X_feat_list, y_list = [], [], []
        n_collected = 0

        for b_idx, (x, y) in enumerate(ds):
            if self.max_batches is not None and b_idx >= self.max_batches:
                break

            if not isinstance(x, dict):
                raise ValueError("Expected dataset elements like ({'images','features'}, y).")

            images_b = x["images"].numpy()
            features_b = x["features"].numpy()
            labels_b = y.numpy()

            remaining = self.max_samples - n_collected
            if remaining <= 0:
                break

            if images_b.shape[0] > remaining:
                images_b = images_b[:remaining]
                features_b = features_b[:remaining]
                labels_b = labels_b[:remaining]

            X_img_list.append(images_b)
            X_feat_list.append(features_b)
            y_list.append(labels_b)

            n_collected += images_b.shape[0]
            if n_collected >= self.max_samples:
                break

        if n_collected == 0:
            raise RuntimeError("No samples collected for SHAP explanation set.")

        self._images = np.concatenate(X_img_list, axis=0)
        self._features = np.concatenate(X_feat_list, axis=0)
        self._labels = np.concatenate(y_list, axis=0)

        # assume validation_ds built from validation_df with shuffle=False
        self._raw_batch_df = (
            self.validation_df.iloc[:n_collected][self.feat_cols].reset_index(drop=True)
        )

    # ---------- Keras hooks ---------- #
    def on_train_begin(self, logs=None):
        self._freeze_batch_from_dataset(self.data_source)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch % self.every_n_epochs) != 0:
            return
        if self._images is None or self._features is None:
            return

        np.random.seed(self.rng_seed)

        # SHAP on frozen set with current weights
        stats, shap_vals, preds = self._engine.explain(
            validation_data=(self._features, self._images, self._labels),
            model=self.model,
            class_index=0,
        )

        mean_abs_all = stats["mean_abs_all"]
        mean_abs_pos = stats["mean_abs_pos"]
        mean_abs_neg = stats["mean_abs_neg"]

        global_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_all
        )
        pos_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_pos
        )
        neg_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_neg
        )

        epoch_name = f"epoch_{epoch:03d}"
        epoch_dir = self.base_dir / epoch_name
        epoch_dir.mkdir(parents=True, exist_ok=True)

        # 1) grouped barplot: global vs pos vs neg (top-k)
        plot_topk_grouped_raw_global_pos_neg(
            global_imp=global_importance_raw,
            pos_imp=pos_importance_raw,
            neg_imp=neg_importance_raw,
            epoch_num = epoch_name,
            k=self.top_k,
            figsize=(10, 5),
            out_path=str(
                epoch_dir
                / f"shap_grouped_raw_top{self.top_k:02d}_glob_pos_neg_{epoch_name}.png"
            ),
        )

        # 2) scatter grid: top-4 numeric features
        plot_shap_scatter_top4_numeric(
            shap_vals=shap_vals,
            preds=preds,
            encoded_feature_names=self.encoded_feature_names,
            global_importance_raw=global_importance_raw,
            raw_batch_df=self._raw_batch_df,
            explained_indices=stats["explained_indices"],
            positive_threshold=self.positive_threshold,
            epoch_num = epoch_name,
            out_path=str(
                epoch_dir / f"shap_top4_numeric_grid_{epoch_name}.png"
            ),
        )

In [66]:
XAI_SHAP_PATH = os.path.join(MODEL_PATH, "xai", "shap_callback")
os.makedirs(XAI_SHAP_PATH, exist_ok=True)

shap_cb = ShapCallback(
    data_source=validation_ds,
    validation_df=validation_df,
    feat_cols=FEAT_COLS,
    encoded_feature_names=encoded_feature_names,
    base_dir=XAI_SHAP_PATH,
    background_size=256,
    max_explain_samples=512,   # how many points SHAP will actually explain
    top_k=15,
    positive_threshold=0.5,
    every_n_epochs=1,
    max_samples=512,           # freeze up to n val samples
    max_batches=20,            # read at most n batches from validation_ds
    rng_seed=0,
)

In [62]:
XAI_SHAP_PATH

'/content/drive/MyDrive/WAIT/models/enhanced/xai/shap_callback'

In [63]:
model_fpath = os.path.join(MODEL_PATH, 'best_enhanced_model_test.keras')

ckpt_cb = keras.callbacks.ModelCheckpoint(
    model_fpath,              # Filepath where the model will be saved.
    monitor="val_auc",        # Metric to monitor (validation AUC in this case).
    save_best_only=True,      # Save only the model with the best performance.
    save_weights_only=False,  # Save the entire model (not just the weights).
    mode="max",               # The model with the maximum 'val_auc' will be saved.
)

In [67]:
cbs = [lr_cb, ckpt_cb, shap_cb]

history = model.fit(
    training_ds,
    epochs=CFG.epochs,
    callbacks=cbs,
    validation_data=validation_ds,
    verbose=CFG.verbose,
    class_weight=class_weights,
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 281s 16s/step - auc: 1.0000 - loss: 0.0658 - val_auc: 0.9316 - val_loss: 0.5866 - learning_rate: 2.5000e-05
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 268s 16s/step - auc: 0.9999 - loss: 0.0689 - val_auc: 0.9208 - val_loss: 0.5967 - learning_rate: 4.4333e-04
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 267s 16s/step - auc: 1.0000 - loss: 0.0634 - val_auc: 0.9246 - val_loss: 0.6299 - learning_rate: 8.6167e-04
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 266s 16s/step - auc: 1.0000 - loss: 0.0640 - val_auc: 0.9008 - val_loss: 0.6586 - learning_rate: 0.0013
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 266s 16s/step - auc: 0.9999 - loss: 0.0673 - val_auc: 0.9184 - val_loss: 0.5876 - learning_rate: 9.6200e-04
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 268s 16s/step - auc: 0.9998 - loss: 0.0691 - val_auc: 0.9277 - val_loss: 0.5096 - learning_rate: 7.2350e-04
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 269s 16s/step - auc: 0.9999 - loss: 0.0686 - val_auc: 0.9374 - val_loss: 0.6269 - learning